# Fine-tuning Text-to-Gloss (T2G) baseline model on PHOENIX-2014T
This notebook replicates the baseline Text-to-Gloss translation model fine-tuning as described in the **WSLP 2025** paper *Finetuning Pre-trained Language Models for Bidirectional Sign Language Gloss to Text Translation*.

### Workflow:
1. Extract annotation gzip splits from the dataset directory.
2. Install training requirements (`transformers`, `evaluate`, `sacrebleu`, `rouge-score`).
3. Train a Sequence-to-Sequence (T5) model to translate German sentences into Uppercase Glosses.
4. Evaluate training using BLEU and ROUGE metric baselines.

In [ ]:
# 1. Unzip annotation files
!mkdir -p /kaggle/working/phoenix_annotations
!cp /kaggle/input/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt/*.gzip /kaggle/working/phoenix_annotations/
!gunzip -S .gzip /kaggle/working/phoenix_annotations/*.gzip
!ls -la /kaggle/working/phoenix_annotations

In [ ]:
# 2. Clone repository to access the training scripts
!git clone https://github.com/Phan-Trung-Thuan/german-VSL /kaggle/working/german-VSL
%cd /kaggle/working/german-VSL

In [ ]:
# 3. Install required training libraries
!pip install -q transformers datasets evaluate sacrebleu rouge-score accelerate sentencepiece

## Start Fine-tuning
We train a baseline Sequence-to-Sequence model using `train_t2g.py`. Feel free to adjust the epochs and model selection (e.g. `google/t5-v1_1-base` or `facebook/mbart-large-50`).

In [ ]:
# 4. Launch Fine-tuning
!python train_t2g.py \
    --base_path /kaggle/working/phoenix_annotations \
    --model_name google/t5-v1_1-base \
    --epochs 5 \
    --batch_size 8 \
    --output_dir /kaggle/working/t2g_model

## Inference Test
Test the trained Text-to-Gloss translation model on custom text sentences.

In [ ]:
# 5. Test Inference
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('/kaggle/working/t2g_model')
model = AutoModelForSeq2SeqLM.from_pretrained('/kaggle/working/t2g_model')

input_text = 'translate German to Gloss: und nun die wettervorhersage fuer morgen donnerstag .'
inputs = tokenizer(input_text, return_tensors='pt', max_length=128, truncation=True)

outputs = model.generate(**inputs, max_length=128)
print('Predicted Gloss:', tokenizer.decode(outputs[0], skip_special_tokens=True).upper())